# obj1b · Teste geral dos GEMs das não-*Saccharomyces* (CarveFungi)

**TASK 3.1** — comunidades microbianas para vinho de baixo teor alcoólico

| código | espécie | ficheiro |
|---|---|---|
| `Td` | *Torulaspora delbrueckii* | `model_delbrueckii.xml` |
| `Mp` | *Metschnikowia pulcherrima* | `model_pulcherrima.xml` |
| `Lt` | *Lachancea thermotolerans* | `model_thermotolerans.xml` |

**Os mesmos catorze testes do `obj1a`, com os mesmos limiares.** Os três modelos
são reconstruções automáticas e precisam de mais rastreio do que o yeast-GEM, que
é curado à mão há mais de dez anos — não de menos. Correr os três lado a lado tem
uma vantagem extra: um defeito que aparece igual nos três é artefacto do template
do CarveFungi; um que aparece só num é propriedade daquele organismo. Essa
distinção decide se a correção vai para a curadoria ou para as limitações.

**A regra é a mesma:** primeiro mede-se, depois decide-se o que curar.

> Alterações de 2026-09-24:
>
> - cinco testes novos, T10 a T14, iguais aos do `obj1a`;
> - as células 20 e 21 deixaram de **alterar** os modelos. Alteravam as fórmulas
>   do C18 e seis estequiometrias de protões fora da secção 11, com `EDICOES`
>   vazio e o log a dizer "0 edicoes": os ficheiros exportados como curados já
>   levavam essas edições e o notebook não as declarava. Passam a ser diagnóstico,
>   e a correção é a edição `NS0` do catálogo, aplicada na secção 11;
> - secção 11 reescrita sobre `scripts/curadoria_minima.py`, catálogo novo
>   derivado só destes testes. **Não usa nem herda nada de `scripts/curadoria.py`**;
> - a variante anaeróbia deixa de ser um segundo ficheiro e passa a ser a função
>   `CU.modo_anaerobio(m, cod)`, pela mesma razão que o yeast-GEM tem um
>   `anaerobicModel.m` e não um `yeast-GEM_anaerobic.xml`.

## 1 · Preparação

In [1]:
import sys, warnings, os
from pathlib import Path
import cobra, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_colwidth", 80)

def raiz(p=None):
    p = (Path(p) if p else Path.cwd()).resolve()
    for c in (p, *p.parents):
        if (c / "models").is_dir() and (c / "scripts").is_dir():
            return c
    raise RuntimeError("raiz do projecto nao encontrada")

ROOT = raiz()
sys.path.insert(0, str(ROOT / "scripts"))
import testes as T
import mapeamento as MP

_lic = os.path.expanduser("~/gurobi.lic")
if os.path.exists(_lic):
    os.environ["GRB_LICENSE_FILE"] = _lic

OUT = ROOT / "results" / "obj1b"
OUT.mkdir(parents=True, exist_ok=True)

CODIGOS  = {"Td": "Td_auto", "Mp": "Mp_auto", "Lt": "Lt_auto"}
ESPECIE  = {"Td": "Torulaspora delbrueckii", "Mp": "Metschnikowia pulcherrima",
            "Lt": "Lachancea thermotolerans"}
FICHEIRO = {"Td": "model_delbrueckii.xml", "Mp": "model_pulcherrima.xml",
            "Lt": "model_thermotolerans.xml"}

MODELOS = {}
for cod, alias in CODIGOS.items():
    MODELOS[cod] = cobra.io.read_sbml_model(str(ROOT / "models" / FICHEIRO[cod]))
    print(f"{cod} ({ESPECIE[cod]}): {len(MODELOS[cod].reactions)} reaccoes")

MAPA = {cod: MP.CROSS_MAP[alias] for cod, alias in CODIGOS.items()}
BIOMASSA = {cod: MAPA[cod]["biomass"] for cod in CODIGOS}   # BIOMASS nos tres
NGAM     = {cod: MAPA[cod]["ngam"] for cod in CODIGOS}

Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Td (Torulaspora delbrueckii): 1970 reaccoes
Mp (Metschnikowia pulcherrima): 2131 reaccoes
Lt (Lachancea thermotolerans): 2138 reaccoes


In [2]:
# Registo dos achados. Cada linha e um defeito MEDIDO, com o teste que o mostrou
# e o modelo onde aparece. E esta lista que autoriza uma edicao de curadoria.
ACHADOS = []

def achado(cod, teste, o_que, gravidade="rever"):
    ACHADOS.append({"codigo": cod, "teste": teste, "achado": o_que,
                    "gravidade": gravidade})
    print(f"  [{cod} {teste}] {o_que}")

## 2 · T1 — o que há nos ficheiros

Se as três colunas forem quase iguais, os três ficheiros vêm do mesmo gerador com
o mesmo template, e o que se encontrar a seguir é do CarveFungi e não da espécie.

In [3]:
print(pd.DataFrame({cod: T.t1_contar(m) for cod, m in MODELOS.items()}).T.to_string())

                 modelo reaccoes metabolitos genes          compartimentos trocas metabolitos_sem_formula reaccoes_sem_GPR
Td  Universal_Universal     1970        1510   487     c, e, l, m, n, r, x    240                      73              767
Mp  Universal_Universal     2131        1633   483  c, e, g, l, m, n, r, x    254                      75              814
Lt  Universal_Universal     2138        1647   491     c, e, l, m, n, r, x    252                      75              803


Existem cerca de 73-75 metabolitos sem fórmulo nos três modelos. **INSPECIONAR!!**

## 3 · T2 e T3 — crescem como vêm, e com tudo aberto

O µ com tudo aberto não é uma taxa de crescimento: sem tectos de captação o modelo
consome o que quiser. Serve só para separar "modelo partido" de "meio apertado".

In [4]:
linhas = []
for cod, m in MODELOS.items():
    r2 = T.t2_crescimento(m)
    linhas.append({"codigo": cod, "mu_meio_do_ficheiro": r2["mu_meio_do_ficheiro"],
                   "trocas_abertas": r2["trocas_abertas_a_captacao"],
                   "mu_tudo_aberto": T.t3_tudo_aberto(m)})
d23 = pd.DataFrame(linhas).set_index("codigo")
print(d23.to_string())

for cod, l in d23.iterrows():
    if l.mu_tudo_aberto < 1e-6:
        achado(cod, "T3", "nao cresce nem com tudo aberto: defeito estrutural", "grave")
    elif l.mu_meio_do_ficheiro < 1e-6:
        achado(cod, "T2", "nao cresce no meio do ficheiro, cresce com tudo aberto: "
                          "o meio do ficheiro esta restrito")

        mu_meio_do_ficheiro  trocas_abertas  mu_tudo_aberto
codigo                                                     
Td                 0.956751              10      102.425041
Mp                 1.084737              10      102.816347
Lt                 0.956398              10      109.258633


O crescimento obtido é irrealista. A M.pulcherrima é das mais lentas e é a que dá valor mais alto. 

### INSPECIONAR CRESCIMENTO OBTIDO

In [5]:
for cod, m in MODELOS.items():
    print(f"\n{cod}")
    for r in sorted(m.exchanges, key=lambda r: r.id):
        if r.lower_bound < 0:
            met = next(iter(r.metabolites))
            print(f"   {r.id:12s} {str(met.name)[:30]:32s} lb={r.lower_bound:g}")


Td
   UF02549_E    ash 1 g                          lb=-1000
   UF02765_E    hydrogenphosphate                lb=-1000
   UF03268_E    iron(2+)                         lb=-1000
   UF03288_E    alpha-D-glucose                  lb=-10
   UF03314_E    water                            lb=-1000
   UF03376_E    ammonium                         lb=-1000
   UF03382_E    dioxygen                         lb=-1000
   UF03456_E    sulfate                          lb=-1000
   UF03474_E    H+                               lb=-1000
   UF03474_TCE  H+                               lb=-1000

Mp
   UF02549_E    ash 1 g                          lb=-1000
   UF02765_E    hydrogenphosphate                lb=-1000
   UF03268_E    iron(2+)                         lb=-1000
   UF03288_E    alpha-D-glucose                  lb=-10
   UF03314_E    water                            lb=-1000
   UF03376_E    ammonium                         lb=-1000
   UF03382_E    dioxygen                         lb=-1000
   UF03456

Este crescimento é justificável pelos bounds abertos default. Não significa nada.

## 4 · T4 — sistema fechado

Com todas as fronteiras fechadas, µ = 0 e ATP = 0. Qualquer valor positivo
significa que o modelo fabrica matéria ou energia do nada, e que **todos** os µ
que ele der estão inflacionados (Fritzemeier et al. 2017).

In [6]:
d4 = pd.DataFrame({cod: T.t4_sistema_fechado(m, NGAM[cod])
                   for cod, m in MODELOS.items()}).T
print(d4.to_string())

for cod, l in d4.iterrows():
    if l.get("biomassa_do_nada", 0) > 1e-6:
        achado(cod, "T4", "cresce com tudo fechado: fuga de massa", "grave")
    if l.get("ATP_do_nada", 0) > 1e-6:
        achado(cod, "T4", f"produz {l['ATP_do_nada']:.3f} ATP com tudo fechado: "
                          f"ciclo gerador de energia", "grave")

    biomassa_do_nada  ATP_do_nada
Td               0.0          0.0
Mp               0.0          0.0
Lt               0.0          0.0


### 4.1 · T10 — ciclos geradores de energia, por transportador

O T4 testa a biomassa e o ATP. Não chega: uma fuga de redox não aparece no teste
do ATP se faltar outro precursor, e um ciclo de protões não aparece em nenhum dos
dois. Nestes modelos interessa sobretudo o quinol e o gradiente mitocondrial, que
é por onde passa a cadeia respiratória.

In [7]:
d10 = pd.DataFrame({cod: T.t10_egc(m, T.PARES_NS) for cod, m in MODELOS.items()}).T
print(d10.to_string())

for cod, linha in d10.iterrows():
    for nome, v in linha.items():
        if v is not None and v > 1e-6:
            achado(cod, "T10", f"ciclo gerador de energia em {nome}: {v:.4f}", "grave")

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp0qa2psub.lp


Reading time = 0.00 seconds
: 1510 rows, 3940 columns, 15096 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpcyaiuuuf.lp
Reading time = 0.01 seconds
: 1510 rows, 3940 columns, 15096 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp1fwqmrq4.lp
Reading time = 0.00 seconds
: 1510 rows, 3940 columns, 15096 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpicmiqin8.lp
Reading time = 0.00 seconds
: 1510 rows, 3940 columns, 15096 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp8mv815if.lp
Reading time = 0.00 seconds
: 1510 rows, 3940 columns, 15096 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmptgbvrni7.lp
Reading time = 0.00 seconds
: 1510 rows, 3940 columns, 15096 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp5fs5yf6f.lp
Reading ti

## 5 · T5 — balanços de massa

As que atravessam a membrana plasmática são as graves: aí o desequilíbrio fabrica
matéria à custa do meio. A coluna `sem_formula` separa o erro real da falta de
informação — estes modelos têm dezenas de metabolitos sem fórmula.

In [8]:
D5, res5 = {}, []
for cod, m in MODELOS.items():
    d = T.t5_balancos(m)
    reais = d[~d.sem_formula] if len(d) else d
    D5[cod] = d
    res5.append({"codigo": cod, "total": len(d),
                 "sem_formula": int(d.sem_formula.sum()) if len(d) else 0,
                 "desequilibrio_real": len(reais),
                 "tocam_extracelular": int(reais.toca_extracelular.sum()) if len(reais) else 0})
    if len(d):
        d.to_csv(OUT / f"T5_balancos_{cod}.csv", index=False)
print(pd.DataFrame(res5).set_index("codigo").to_string())

for cod in MODELOS:
    d = D5[cod]
    reais = d[~d.sem_formula] if len(d) else d
    if len(reais) and reais.toca_extracelular.any():
        print(f"\n{cod} — atravessam a membrana plasmatica:")
        print(reais[reais.toca_extracelular][["rxn","nome","compartimentos","desvio"]]
              .to_string(index=False))
        achado(cod, "T5", f"{int(reais.toca_extracelular.sum())} reaccoes "
                          f"desequilibradas atravessam a membrana", "grave")

        total  sem_formula  desequilibrio_real  tocam_extracelular
codigo                                                            
Td        100           50                  50                   0
Mp        119           54                  65                   0
Lt        112           53                  59                   0


### 5.1 · Fórmulas dos ácidos gordos

Um desequilíbrio de membrana nestes modelos costuma vir das fórmulas dos ácidos
gordos. O anião do estearato (C18:0) é C18H35O2⁻ e o do oleato (C18:1) é
C18H33O2⁻: a diferença de 2 H é o que distingue saturado de insaturado. Se as duas
estiverem anotadas iguais, a dessaturação fica elementarmente nula e o modelo
fabrica insaturados sem O₂ e sem dessaturase.

Os pares C16 e C14 servem de controlo interno: se estiverem certos e o C18 errado,
é um erro isolado e não uma convenção do ficheiro.

In [9]:
for cod, m in MODELOS.items():
    print(f"\n{cod}:")
    for mid in ("ocda", "9Zocda", "hxda", "9Zhxda", "tda"):
        for s in [x for x in m.metabolites if x.id.startswith(mid)][:2]:
            print(f"   {s.id:16s} {str(s.name)[:30]:32s} {str(s.formula):12s} carga={s.charge}")


Td:
   ocda_x           stearate                         C18H34O2     carga=-1
   ocda_c           stearate                         C18H34O2     carga=-1
   9Zocda_x         (9Z)-oleate                      C18H34O2     carga=-1
   9Zocda_c         (9Z)-oleate                      C18H34O2     carga=-1
   hxda_x           palmitate                        C16H31O2     carga=-1
   hxdal_e          hexadecanal                      C16H32O      carga=0
   9Zhxda_c         (9Z)-hexadecenoate               C16H29O2     carga=-1
   9Zhxda_e         (9Z)-hexadecenoate               C16H29O2     carga=-1
   tda_c            tetradecanoate                   C14H27O2     carga=-1
   tda_e            tetradecanoate                   C14H27O2     carga=-1

Mp:
   ocda_x           stearate                         C18H34O2     carga=-1
   ocda_c           stearate                         C18H34O2     carga=-1
   9Zocda_x         (9Z)-oleate                      C18H34O2     carga=-1
   9Zocda_c     

Nos três modelos, ocda (estearato) e 9Zocda (oleato) estão ambos como C18H34O2 com carga −1. Nenhum está certo: os aniões são C18H35O2⁻ e C18H33O2⁻. **INSPECIONAR**

In [10]:
# DIAGNOSTICO, nao correccao. A correccao e a edicao NS0 da seccao 11.
# Antes, estas duas celulas alteravam os modelos aqui, fora da seccao de
# curadoria: os ficheiros exportados como curados ja levavam a correccao e o
# log da seccao 11 dizia "0 edicoes".
CORRIGIR_FORMULA = {"ocda": ("C18H35O2", -1),    # estearato, C18:0
                    "9Zocda": ("C18H33O2", -1)}  # oleato,    C18:1

for cod, m in MODELOS.items():
    for base, (formula, carga) in CORRIGIR_FORMULA.items():
        for s in [x for x in m.metabolites
                  if x.id == base or x.id.startswith(base + "_")]:
            if s.formula != formula:
                print(f"{cod} {s.id:12s} {s.formula} deveria ser {formula}")

Td ocda_x       C18H34O2 deveria ser C18H35O2
Td ocda_c       C18H34O2 deveria ser C18H35O2
Td ocda_e       C18H34O2 deveria ser C18H35O2
Td 9Zocda_x     C18H34O2 deveria ser C18H33O2
Td 9Zocda_c     C18H34O2 deveria ser C18H33O2
Td 9Zocda_e     C18H34O2 deveria ser C18H33O2
Mp ocda_x       C18H34O2 deveria ser C18H35O2
Mp ocda_c       C18H34O2 deveria ser C18H35O2
Mp ocda_e       C18H34O2 deveria ser C18H35O2
Mp 9Zocda_x     C18H34O2 deveria ser C18H33O2
Mp 9Zocda_c     C18H34O2 deveria ser C18H33O2
Mp 9Zocda_e     C18H34O2 deveria ser C18H33O2
Lt ocda_x       C18H34O2 deveria ser C18H35O2
Lt ocda_c       C18H34O2 deveria ser C18H35O2
Lt ocda_e       C18H34O2 deveria ser C18H35O2
Lt 9Zocda_x     C18H34O2 deveria ser C18H33O2
Lt 9Zocda_c     C18H34O2 deveria ser C18H33O2
Lt 9Zocda_e     C18H34O2 deveria ser C18H33O2


In [11]:
# As reaccoes cujo balanco depende das formulas do C18. Com as formulas erradas
# aparecem equilibradas; com as formulas certas ficam com desvio de 1 protao.
# So se mede aqui; corrige-se na NS0.
ALVOS = ["UF01585_X", "UF01591_C", "UF02328_C", "UF02328_X", "UF02342_C", "UF02342_X"]

linhas = []
for cod, m in MODELOS.items():
    prova = m.copy()                      # copia: os modelos nao se alteram aqui
    for base, (formula, carga) in CORRIGIR_FORMULA.items():
        for s in [x for x in prova.metabolites
                  if x.id == base or x.id.startswith(base + "_")]:
            s.formula, s.charge = formula, carga
    for rid in ALVOS:
        if rid not in prova.reactions:
            linhas.append({"cod": cod, "rxn": rid, "desvio_com_formula_certa": "nao existe"})
            continue
        r = prova.reactions.get_by_id(rid)
        d = {k: v for k, v in r.check_mass_balance().items()
             if k != "charge" and abs(v) > 1e-6}
        linhas.append({"cod": cod, "rxn": rid, "equacao": r.reaction[:58],
                       "desvio_com_formula_certa": d or "equilibrada"})
print(pd.DataFrame(linhas).to_string(index=False))

achado("Td", "T5.1", "formulas do par C18 iguais (ocda e 9Zocda ambos C18H34O2); "
                     "6 reaccoes com desvio de 1 protao dependem delas", "grave")
achado("Mp", "T5.1", "idem", "grave")
achado("Lt", "T5.1", "idem", "grave")

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpvuoxlp1l.lp
Reading time = 0.00 seconds
: 1510 rows, 3940 columns, 15096 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmptx75vtlj.lp
Reading time = 0.01 seconds
: 1633 rows, 4262 columns, 16440 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpa11c1zgj.lp
Reading time = 0.00 seconds
: 1647 rows, 4276 columns, 16484 nonzeros
cod       rxn                                                    equacao desvio_com_formula_certa
 Td UF01585_X               h2o_x + stcoa_x --> coa_x + 2.0 h_x + ocda_x               {'H': 1.0}
 Td UF01591_C                    9Zocdcoa_c + h2o_c --> 9Zocda_c + coa_c              {'H': -1.0}
 Td UF02328_C 9Zocda_c + atp_c + coa_c --> 9Zocdcoa_c + amp_c + h_c + pp               {'H': 1.0}
 Td UF02328_X 9Zocda_x + atp_x + coa_x --> 9Zocdcoa_x + amp_x + h_x + pp               {'H': 1.0}
 Td UF02342_C  

## 6 · T6 — a biomassa

A escala de µ e o custo de crescer. Se as três reações de biomassa tiverem a mesma
assinatura, a composição vem do template e não da espécie — é limitação a
declarar, não resultado.

In [12]:
R6, F6 = {}, {}
for cod, m in MODELOS.items():
    R6[cod], F6[cod] = T.t6_biomassa(m, BIOMASSA[cod])
print(pd.DataFrame(R6).T.to_string())


    reaccao n_reagentes massa_por_formula_g componentes_sem_massa soma_coef_componentes_sem_massa soma_coef_1g GAM_mmol_ATP_por_gDW                  escala desvio_de_1g_pct
Td  BIOMASS          22             30.1894                     0                             0.0          1.0                 60.0  pseudo-componentes _1g              0.0
Mp  BIOMASS          22             30.1894                     0                             0.0          1.0                 60.0  pseudo-componentes _1g              0.0
Lt  BIOMASS          22             30.1894                     0                             0.0          1.0                 60.0  pseudo-componentes _1g              0.0


In [13]:
# A equação de biomassa de cada modelo, termo a termo.
for cod, m in MODELOS.items():
    bio = m.reactions.get_by_id(BIOMASSA[cod])
    print(f"\n{'='*80}\n{cod} — {ESPECIE[cod]}  |  {bio.id}\n{'='*80}")
    
    # 1. Imprime a equação completa por escrito
    print(f"Equação: {bio.reaction}\n{'-'*80}")
    
    # 2. Constrói e imprime a tabela termo a termo
    linhas = []
    for met, coef in sorted(bio.metabolites.items(), key=lambda x: x[1]):
        linhas.append({"lado": "consome" if coef < 0 else "produz",
                       "coef": round(coef, 6), "id": met.id,
                       "nome": (met.name or "")[:34], "formula": met.formula})
    print(pd.DataFrame(linhas).to_string(index=False))


Td — Torulaspora delbrueckii  |  BIOMASS
Equação: 0.253 13BDglcn_1g_c + 0.048 ash_1g_c + 60.0 atp_c + 0.0001 btn_c + 0.01 chitin_1g_c + 0.0001 coa_c + 0.005 dna_1g_c + 0.0001 ficytc_c + 0.0001 focytc_c + 0.1 glycogen_1g_c + 60.0 h2o_c + 0.07 mannan_1g_c + 0.04 phospholipid_1g_c + 0.4 protein_1g_c + 0.0001 q6_m + 0.0001 retn_c + 0.0001 ribflv_c + 0.058 rna_1g_c + 0.001 sterol_1g_r + 0.0001 thf_c + 0.0001 thmtp_c + 0.015 trehalose_1g_c --> 60.0 adp_c + biomass_1g_c + 60.0 h_c + 60.0 pi_c
--------------------------------------------------------------------------------
   lado     coef                id                          nome        formula
consome -60.0000             atp_c                       ATP(4-)  C10H12N5O13P3
consome -60.0000             h2o_c                         water            H2O
consome  -0.4000      protein_1g_c                       protein       C2H4NO2R
consome  -0.2530     13BDglcn_1g_c      (1->3)-beta-D-glucan 1 g           None
consome  -0.1000     glycog

Os três modelos possuem exatamente a mesma reação de biomassa - três espécies diferentes com a mesma composição celular -> declarar nos resultados.

## 7 · T7 — meio mínimo, com o açúcar fixo

Fixando a glucose e a frutose e fechando as outras fontes de carbono, pergunta-se:
*com o açúcar do mosto, de que mais é que cada modelo precisa?*

O pseudo-metabolito `ash` (1 g de minerais) fica aberto: não tem fórmula, logo não
se consegue excluir que traga carbono, mas fechá-lo bloqueia os três modelos. É a
limitação já conhecida destes ficheiros — no eixo mineral não são comparáveis com
a *S. cerevisiae*, que troca K, Mg, Ca, Mn e Zn separadamente.

In [14]:
MEIO_MIN = {}
for cod, m in MODELOS.items():
    acucares = [MAPA[cod]["entradas"][k] for k in ("glucose", "fructose")
                if k in MAPA[cod]["entradas"]]
    manter = [MAPA[cod]["entradas"]["ash"]] if "ash" in MAPA[cod]["entradas"] else []
    print(f"\n{cod}:")
    meio, mu = T.t7_meio_minimo(m, fraccao=0.1, fontes_de_C=acucares,
                                q_fonte=10.0, manter_abertas=manter)
    MEIO_MIN[cod] = meio
    if meio is None:
        achado(cod, "T7", "nao cresce com glucose e frutose como unicas fontes "
                          "de carbono", "grave")
        continue
    tab = T.descrever_meio(m, meio)
    print(f"   mu maximo {mu:.4f} h-1 | meio minimo: {len(tab)} trocas")
    print(tab.to_string(index=False))
    tab.to_csv(OUT / f"T7_meio_minimo_{cod}.csv", index=False)


Td:
   2 trocas fechadas por o metabolito nao ter formula: UF03052_E (chitosan), UF03299_E (glycogen)
   mu maximo 1.9158 h-1 | meio minimo: 7 trocas
    troca      metabolito formula  captacao
UF03275_E      D-fructose C6H12O6    5.1673
UF03382_E        dioxygen      O2    2.4373
UF03376_E        ammonium     NH4    1.0595
UF03288_E alpha-D-glucose C6H12O6    0.4241
UF03455_E         sulfite     SO3    0.0108
UF02549_E         ash 1 g    None    0.0092
UF03268_E        iron(2+)      Fe    0.0000

Mp:
   2 trocas fechadas por o metabolito nao ter formula: UF03052_E (chitosan), UF03299_E (glycogen)
   mu maximo 2.1720 h-1 | meio minimo: 7 trocas
    troca      metabolito formula  captacao
UF03275_E      D-fructose C6H12O6    4.5559
UF03382_E        dioxygen      O2    2.2019
UF03376_E        ammonium     NH4    1.2012
UF03288_E alpha-D-glucose C6H12O6    0.4808
UF03455_E         sulfite     SO3    0.0123
UF02549_E         ash 1 g    None    0.0104
UF03268_E        iron(2+)      Fe    0

## 8 · Meio de trabalho declarado

O mesmo desenho do `obj1a`: açúcar, azoto, minerais, vitaminas e os suplementos da
auxotrofia anaeróbia, todos com tecto declarado. As vitaminas e os esteróis têm
carbono — abertos sem tecto tornam-se fontes de carbono e o µ deixa de medir o que
se pensa que mede.

Nos três CarveFungi a lista `anaerobio` do `mapeamento.py` tem só o ergosterol
(`UF03007_E`). O oleato (`UF02973_E`) e o palmitoleato (`UF02972_E`) existem nas
entradas e são abertos aqui, pela mesma razão do `obj1a`: em anaerobiose a
levedura não sintetiza ácidos gordos insaturados.

In [15]:
Q_ACUCAR, Q_N, Q_VITAMINA, Q_SUPLEMENTO = 10.0, 5.0, 0.05, 0.05
SUPLEMENTOS = ("ergosterol", "oleate", "palmitoleate")

def meio_de_trabalho(m, cod, q_o2=0.0):
    """Fecha toda a captacao e abre um meio declarado. Altera in place."""
    ent = MAPA[cod]["entradas"]
    pega = lambda nomes: [ent[k] for k in nomes if k in ent]
    T.fechar_captacao(m)
    T.abrir(m, pega(("glucose", "fructose")), Q_ACUCAR)
    T.abrir(m, pega(("ammonium",)), Q_N)
    T.abrir(m, pega(MP.INORGANICOS + ("ash",)), 1000.0)      # nao trazem carbono
    T.abrir(m, pega(("water", "proton", "co2")), 1000.0)
    T.abrir(m, pega(MP.VITAMINAS), Q_VITAMINA)               # tem carbono: com tecto
    T.abrir(m, pega(SUPLEMENTOS), Q_SUPLEMENTO)              # idem
    T.abrir(m, pega(("oxygen",)), q_o2)
    return m

with MODELOS["Td"] as m:
    meio_de_trabalho(m, "Td", q_o2=0.0)
    for r in sorted((r for r in m.exchanges if r.lower_bound < 0), key=lambda r: r.id):
        met = next(iter(r.metabolites))
        print(f"   {r.id:12s} {str(met.name)[:34]:36s} lb={r.lower_bound:8g}")

   UF02549_E    ash 1 g                              lb=   -1000
   UF02765_E    hydrogenphosphate                    lb=   -1000
   UF02766_E    (R)-pantothenate                     lb=   -0.05
   UF02775_E    pyridoxine                           lb=   -0.05
   UF02972_E    (9Z)-hexadecenoate                   lb=   -0.05
   UF02973_E    (9Z)-oleate                          lb=   -0.05
   UF02997_E    biotinate                            lb=   -0.05
   UF03007_E    ergosterol                           lb=   -0.05
   UF03227_E    carbon dioxide                       lb=   -1000
   UF03268_E    iron(2+)                             lb=   -1000
   UF03275_E    D-fructose                           lb=     -10
   UF03288_E    alpha-D-glucose                      lb=     -10
   UF03314_E    water                                lb=   -1000
   UF03344_E    myo-inositol                         lb=   -0.05
   UF03373_E    nicotinate                           lb=   -0.05
   UF03376_E    ammonium 

## 9 · T8 e T9 — compostos indispensáveis, e oxigénio

**A secção que decide as pendências.** A `C4a` na Mp e a `CUR6` na Td estavam em
aberto à espera do meio do mosto. Não precisam dele: o que as decide é se o modelo
cresce a O₂ = 0 com esteróis e ácidos gordos insaturados fornecidos, e isso
mede-se aqui.

In [16]:
D8 = {}
for cod, m in MODELOS.items():
    with m:
        meio_de_trabalho(m, cod, q_o2=2.0)
        d = T.t8_essenciais(m)
    D8[cod] = d
    ess = d[d.essencial] if len(d) else d
    print(f"\n{cod}: indispensaveis {len(ess)} de {len(d)}")
    print(d.to_string(index=False))
    d.to_csv(OUT / f"T8_essenciais_{cod}.csv", index=False)
    if len(ess):
        achado(cod, "T8", f"indispensaveis: {', '.join(ess.metabolito)}", "informativo")


Td: indispensaveis 4 de 20
    troca         metabolito  mu_sem_ele  perda_pct  essencial
UF02549_E            ash 1 g    0.000000      100.0       True
UF03268_E           iron(2+)    0.000000      100.0       True
UF03382_E           dioxygen    0.000000      100.0       True
UF03456_E            sulfate    0.000000      100.0       True
UF03376_E           ammonium    0.009043       98.1      False
UF03275_E         D-fructose    0.272049       41.9      False
UF03288_E    alpha-D-glucose    0.272049       41.9      False
UF02972_E (9Z)-hexadecenoate    0.461102        1.6      False
UF02973_E        (9Z)-oleate    0.464660        0.8      False
UF02775_E         pyridoxine    0.466089        0.5      False
UF03007_E         ergosterol    0.467073        0.3      False
UF02765_E  hydrogenphosphate    0.467549        0.2      False
UF03227_E     carbon dioxide    0.468372       -0.0      False
UF03314_E              water    0.468372       -0.0      False
UF03344_E       myo-inosito

In [17]:
D9 = {}
for cod, m in MODELOS.items():
    prod = {k: MAPA[cod]["saidas"][k] for k in
            ("ethanol", "glycerol", "acetate", "L-lactate", "acetaldehyde")
            if k in MAPA[cod]["saidas"]}
    with m:
        meio_de_trabalho(m, cod, q_o2=0.0)
        D9[cod] = T.t9_oxigenio(m, MAPA[cod]["entradas"]["oxygen"],
                                niveis=(0.0, 0.5, 1.0, 2.0, 5.0, 20.0),
                                produtos=prod, bio_rid=BIOMASSA[cod])
    print(f"\n{cod} — {ESPECIE[cod]}")
    print(D9[cod].to_string(index=False))
    D9[cod].to_csv(OUT / f"T9_oxigenio_{cod}.csv", index=False)

    an = D9[cod][D9[cod].O2_disponivel == 0.0].iloc[0]
    if not an.cresce:
        achado(cod, "T9", "NAO cresce em anaerobiose estrita com esterois e UFA "
                          "fornecidos: falta uma via na reconstrucao, ou e fenotipo "
                          "real do organismo", "grave")


Td — Torulaspora delbrueckii
 O2_disponivel       mu  cresce  estado  ethanol  glycerol  acetate  L-lactate  acetaldehyde
           0.0 0.000000   False optimal      0.0       0.0   0.0000     0.0000           0.0
           0.5 0.397001    True optimal      0.0       0.0   0.0000    35.0103           0.0
           1.0 0.424282    True optimal      0.0       0.0   0.0000    34.6208           0.0
           2.0 0.468372    True optimal      0.0       0.0   0.0000     0.0000           0.0
           5.0 0.575614    True optimal      0.0       0.0   0.0000     0.0000           0.0
          20.0 0.913357    True optimal      0.0       0.0   6.5203     6.3996           0.0
  [Td T9] NAO cresce em anaerobiose estrita com esterois e UFA fornecidos: falta uma via na reconstrucao, ou e fenotipo real do organismo

Mp — Metschnikowia pulcherrima
 O2_disponivel       mu  cresce  estado  ethanol  glycerol  acetate  L-lactate  acetaldehyde
           0.0 0.000000   False optimal      0.0       0

## 9.1 · T11 — o que bloqueia a anaerobiose

O T9 diz que nenhum dos três cresce sem O₂. Não diz porquê, e sem isso qualquer
edição é um palpite. O T11 pede a produção máxima de cada reagente da biomassa,
com e sem oxigénio, e desce pelas pseudo-reações dos componentes bloqueados.

Um componente bloqueado nos três é artefacto do template; num só é propriedade
daquele organismo.

In [18]:
D11 = {}
for cod, m in MODELOS.items():
    meio_cod = (lambda c: (lambda mm, q_o2=0.0: meio_de_trabalho(mm, c, q_o2=q_o2)))(cod)
    D11[cod] = T.t11_bloqueio_anaerobio(m, BIOMASSA[cod], meio_cod, maxprof=2)
    D11[cod].to_csv(OUT / f"T11_bloqueio_{cod}.csv", index=False)

d11 = pd.concat(D11, names=["codigo"]).reset_index(level=0)
bloq = d11[d11.so_com_O2]
print(bloq[["codigo", "metabolito", "nome", "vem_de", "max_sem_O2", "max_com_O2"]]
      .to_string(index=False))

for cod in MODELOS:
    b = bloq[bloq.codigo == cod]
    if len(b):
        achado(cod, "T11", "so se fabricam com O2: " +
               ", ".join(f"{r.metabolito}" for r in b.itertuples()), "grave")

nos_tres = (bloq.groupby("metabolito").codigo.nunique() == 3)
print("\nbloqueados nos tres (artefacto do template):",
      ", ".join(sorted(nos_tres[nos_tres].index)))

codigo  metabolito              nome    vem_de  max_sem_O2  max_com_O2
    Td    dna_1g_c           DNA 1 g   BIOMASS         0.0    0.429203
    Td    rna_1g_c           RNA 1 g   BIOMASS         0.0    0.453769
    Td sterol_1g_r            sterol   BIOMASS         0.0    0.086427
    Td    ficytc_c ferricytochrome c   BIOMASS         0.0    0.500000
    Td    focytc_c ferrocytochrome c   BIOMASS         0.0    0.500000
    Td      retn_c         retinoate   BIOMASS         0.0    0.740552
    Td        q6_m      ubiquinone-6   BIOMASS         0.0    0.420602
    Td      dctp_c          dCTP(4-) UF02406_C         0.0    1.683333
    Td      dttp_c          dTTP(4-) UF02406_C         0.0    2.525000
    Td       ctp_c           CTP(4-) UF02414_C         0.0    1.683333
    Td       utp_c           UTP(4-) UF02414_C         0.0    2.525000
    Td   retinal_c           retinal UF00546_C         0.0    0.759913
    Mp    dna_1g_c           DNA 1 g   BIOMASS         0.0    0.429203
    Mp

## 9.2 · T12 — termos residuais da biomassa

Os quatro cofactores da biomassa destes modelos têm coeficiente de 10⁻⁴ e pesam
menos de um décimo de miligrama por grama de células. Podem, ainda assim, decidir
sozinhos se o modelo cresce. A coluna `decide_anaerobiose` é a que interessa.

In [19]:
D12 = {}
for cod, m in MODELOS.items():
    meio_cod = (lambda c: (lambda mm, q_o2=0.0: meio_de_trabalho(mm, c, q_o2=q_o2)))(cod)
    D12[cod] = T.t12_termos_residuais(m, [BIOMASSA[cod]], meio_cod, limiar_coef=1e-3)
    D12[cod].to_csv(OUT / f"T12_termos_{cod}.csv", index=False)

d12 = pd.concat(D12, names=["codigo"]).reset_index(level=0)
print(d12[["codigo", "metabolito", "nome", "coef", "massa_mg_por_gDW",
           "mu_O2_0_com", "mu_O2_0_sem", "mu_O2_2_com", "mu_O2_2_sem",
           "decide_anaerobiose"]].to_string(index=False))

for r in d12[d12.decide_anaerobiose].itertuples():
    achado(r.codigo, "T12", f"{r.metabolito} ({r.nome}): coef {r.coef:.1e} "
                            f"(~{r.massa_mg_por_gDW:.1e} mg/gDW) decide sozinho "
                            f"o crescimento anaerobio", "grave")

codigo  metabolito                         nome    coef  massa_mg_por_gDW  mu_O2_0_com  mu_O2_0_sem  mu_O2_2_com  mu_O2_2_sem  decide_anaerobiose
    Td sterol_1g_r                       sterol -0.0010          0.000000          0.0          0.0     0.468372     0.468422               False
    Td    ficytc_c            ferricytochrome c -0.0001          0.061346          0.0          0.0     0.468372     0.468395               False
    Td    focytc_c            ferrocytochrome c -0.0001          0.061447          0.0          0.0     0.468372     0.468394               False
    Td      retn_c                    retinoate -0.0001          0.029943          0.0          0.0     0.468372     0.468395               False
    Td     thmtp_c thiamine(+) triphosphate(4-) -0.0001          0.050126          0.0          0.0     0.468372     0.468373               False
    Td    ribflv_c               riboflavin(1-) -0.0001          0.037536          0.0          0.0     0.468372     0.46838

## 9.3 · T13 — produtos em intervalo, não em ponto

As colunas de secreção do T9 vêm de pFBA, que escolhe entre ótimos alternativos
pelo número de passos e não pela bioquímica: piruvato → lactato é um passo,
piruvato → etanol são dois, e é só por isso que o lactato ganha. A célula 12.3
mostra que fechar o lactato não muda µ.

O T13 faz FVA no ótimo e devolve (mínimo, máximo). Só um mínimo maior do que zero
autoriza a frase "este modelo produz X".

In [20]:
D13 = {}
for cod, m in MODELOS.items():
    saidas = {k: v for k, v in MAPA[cod]["saidas"].items() if v in m.reactions}
    with m:
        meio_de_trabalho(m, cod, q_o2=0.0)
        D13[cod] = T.t13_produtos(m, MAPA[cod]["entradas"]["oxygen"], saidas,
                                  niveis=(0.0, 0.5, 2.0, 20.0))
    print(f"\n{cod} — {ESPECIE[cod]}")
    print(D13[cod].to_string())
    D13[cod].to_csv(OUT / f"T13_produtos_{cod}.csv")

print("\nmaximo teorico do etanol a partir de 20 mmol de hexose = 40 mmol/gDW/h")
for cod in MODELOS:
    et = D13[cod].loc["ethanol", "O2=2"] if "ethanol" in D13[cod].index else None
    if et and isinstance(et, tuple) and et[1] < 1e-6:
        achado(cod, "T13", "nao consegue produzir etanol a nenhum nivel de O2", "grave")
    elif et and isinstance(et, tuple) and et[0] < 1e-6:
        achado(cod, "T13", f"etanol e alternativo e nao obrigatorio: intervalo "
                           f"({float(et[0]):.2f}, {float(et[1]):.2f}) a O2 = 2; "
                           f"a pFBA do T9 escolhe lactato", "grave")

Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp4mjx1zqt.lp
Reading time = 0.00 seconds
: 1511 rows, 3941 columns, 15099 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpeazvmqvc.lp
Reading time = 0.00 seconds
: 1511 rows, 3941 columns, 15099 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpyn7p4cgs.lp
Reading time = 0.00 seconds
: 1511 rows, 3941 columns, 15099 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use onl

## 9.4 · T14 — balanço de carbono e rendimento em etanol

O critério de aceitação de qualquer edição. Numa fermentação vínica real o etanol
fica a 90–92 % do máximo teórico de 2 mol por mol de hexose, com o glicerol a
consumir 4–6 % do açúcar. O fecho de carbono deve ficar entre 95 e 105 %.

A coluna `trocas_sem_formula` é importante aqui: o `ash 1 g` não tem fórmula, logo
conta zero carbono, e pode estar a esconder desvio.

In [21]:
linhas = []
for cod, m in MODELOS.items():
    for q in (0.0, 2.0, 20.0):
        with m:
            meio_de_trabalho(m, cod, q_o2=q)
            linhas.append({"codigo": cod, "O2": q,
                           **T.t14_carbono(m, BIOMASSA[cod],
                                           MAPA[cod]["saidas"]["ethanol"])})
d14 = pd.DataFrame(linhas)
print(d14.to_string(index=False))
d14.to_csv(OUT / "T14_carbono.csv", index=False)

codigo   O2       mu  cresce  C_entra_mmol  C_sai_mmol  C_biomassa_mmol  fecho_pct  hexose_consumida  etanol  etanol_por_hexose_mol  pct_do_maximo_teorico trocas_sem_formula
    Td  0.0 0.000000   False        120.40      120.40             0.00      100.0             20.00    0.00                  0.000                    0.0                  -
    Td  2.0 0.468372    True        121.15      103.47            19.20      101.3             20.00    0.92                  0.046                    2.3          UF02549_E
    Td 20.0 0.913357    True        122.21       87.74            37.45      102.4             20.01    0.00                  0.000                    0.0          UF02549_E
    Mp  0.0 0.000000   False         92.50       92.50             0.00      100.0             15.42    0.00                  0.000                    0.0                  -
    Mp  2.0 0.506788    True        121.21      102.08            20.78      101.4             20.00    0.00                  0.00

### 9.5 · Rastreio específico dos bloqueios

As células seguintes são o rastreio manual dos componentes que o T11 marcou: quem
os produz, e onde é que a cadeia das pirimidinas se parte. É o que transforma
"este componente está bloqueado" em "esta reação é que falta".

In [22]:
# Para cada componente da biomassa: consegue-se fabricar sem O2?
linhas = []
for cod, m in MODELOS.items():
    bio = m.reactions.get_by_id(BIOMASSA[cod])
    for met in bio.reactants:
        valores = {}
        for q_o2 in (0.0, 2.0):
            with m:
                meio_de_trabalho(m, cod, q_o2=q_o2)
                dreno = m.add_boundary(met, type="demand")
                m.objective = dreno
                v = m.slim_optimize()
            valores[q_o2] = 0.0 if v != v else round(v, 6)
        linhas.append({"codigo": cod, "componente": met.id,
                       "nome": (met.name or "")[:30],
                       "max_sem_O2": valores[0.0], "max_com_O2": valores[2.0],
                       "so_com_O2": valores[0.0] < 1e-9 and valores[2.0] > 1e-9})

d_bloq = pd.DataFrame(linhas)
bloq = d_bloq[d_bloq.so_com_O2]
print(f"componentes que so se fabricam COM oxigenio: {len(bloq)}\n")
print(bloq.to_string(index=False) if len(bloq) else "nenhum")
d_bloq.to_csv(OUT / "bloqueio_anaerobio.csv", index=False)

componentes que so se fabricam COM oxigenio: 21

codigo  componente              nome  max_sem_O2  max_com_O2  so_com_O2
    Td    dna_1g_c           DNA 1 g         0.0    0.429203       True
    Td    rna_1g_c           RNA 1 g         0.0    0.453769       True
    Td sterol_1g_r            sterol         0.0    0.086427       True
    Td    ficytc_c ferricytochrome c         0.0    0.500000       True
    Td    focytc_c ferrocytochrome c         0.0    0.500000       True
    Td      retn_c         retinoate         0.0    0.740552       True
    Td        q6_m      ubiquinone-6         0.0    0.420602       True
    Mp    dna_1g_c           DNA 1 g         0.0    0.429203       True
    Mp    rna_1g_c           RNA 1 g         0.0    0.453769       True
    Mp sterol_1g_r            sterol         0.0    0.086427       True
    Mp    ficytc_c ferricytochrome c         0.0    0.470588       True
    Mp    focytc_c ferrocytochrome c         0.0    0.500000       True
    Mp      ret

In [23]:
def quem_faz(m, met_id, max_reaccoes=6):
    if met_id not in m.metabolites:
        print(f"   {met_id}: nao existe neste modelo"); return
    met = m.metabolites.get_by_id(met_id)
    prod = [r for r in met.reactions if r.metabolites[met] > 0 and not r.boundary]
    print(f"   {met.id} ({met.name}) — {len(prod)} reaccoes que o produzem")
    for r in prod[:max_reaccoes]:
        print(f"      {r.id:14s} {r.reaction[:100]}")
        print(f"      {'':14s} bounds={r.bounds}  GPR={r.gene_reaction_rule or '(sem gene)'}")

# mudar a lista conforme o que o bloco C assinalar
for cod, m in MODELOS.items():
    print(f"\n{'='*84}\n{cod}\n{'='*84}")
    for mid in ("sterol_1g_r", "dna_1g_c", "rna_1g_c", "q6_m"):
        quem_faz(m, mid)


Td
   sterol_1g_r (sterol) — 2 reaccoes que o produzem
      UF02416_R      0.96 cholest_1g_r + 0.04 zymst_1g_r --> sterol_1g_r
                     bounds=(0.0, 1000.0)  GPR=(sem gene)
      UF02415_R      0.96 ergst_1g_r + 0.04 zymst_1g_r --> sterol_1g_r
                     bounds=(0.0, 1000.0)  GPR=(sem gene)
   dna_1g_c (DNA 1 g) — 1 reaccoes que o produzem
      UF02406_C      0.954 datp_c + 0.636 dctp_c + 0.636 dgtp_c + 0.954 dttp_c + h2o_c + 3.179 h_c --> dna_1g_c + 3.179 p
                     bounds=(0.0, 1000.0)  GPR=(sem gene)
   rna_1g_c (RNA 1 g) — 1 reaccoes que o produzem
      UF02414_C      0.716 atp_c + 0.697 ctp_c + 0.718 gtp_c + h2o_c + 3.064 h_c + 0.934 utp_c --> 3.064 ppi_c + rna_1g_c
                     bounds=(0.0, 1000.0)  GPR=(sem gene)
   q6_m (ubiquinone-6) — 2 reaccoes que o produzem
      UF00845_M      2hpmhmbq_m + amet_m --> ahcys_m + 2.0 h_m + q6_m
                     bounds=(0.0, 1000.0)  GPR=CCE92109
      UF02392_M      2.0 ficytc_m + 4.0 h_m + q

In [24]:
# Onde parte a cadeia das pirimidinas. O DNA e o RNA precisam dos nucleotidos;
# os de pirimidina (C, T, U) vem todos do orotato. Este bloco ve ate onde chega.
alvos = ["dhorS_c","dhorS_m","orot_c","orot_m","ump_c","utp_c","ctp_c",
         "dttp_c","dctp_c","atp_c","gtp_c"]

linhas = []
for cod, m in MODELOS.items():
    for mid in alvos:
        if mid not in m.metabolites:
            linhas.append({"cod": cod, "met": mid, "sem_O2": None, "com_O2": None}); continue
        met = m.metabolites.get_by_id(mid); v = {}
        for q in (0.0, 2.0):
            with m:
                meio_de_trabalho(m, cod, q_o2=q)
                m.objective = m.add_boundary(met, type="demand")
                x = m.slim_optimize()
            v[q] = 0.0 if x != x else round(x, 4)
        linhas.append({"cod": cod, "met": mid, "sem_O2": v[0.0], "com_O2": v[2.0]})

print(pd.DataFrame(linhas).pivot(index="met", columns="cod",
                                 values=["sem_O2","com_O2"]).to_string())

print("\nquem produz o orotato:")
for cod, m in MODELOS.items():
    print(f"\n{cod}:")
    for mid in ("orot_c", "orot_m"):
        if mid not in m.metabolites: 
            print(f"   {mid}: nao existe neste modelo"); continue
        met = m.metabolites.get_by_id(mid)
        for r in met.reactions:
            if r.metabolites[met] > 0 and not r.boundary:
                print(f"   {r.id:12s} {r.reaction[:88]}")
                print(f"   {'':12s} GPR={r.gene_reaction_rule or '(sem gene)'}")

        sem_O2                 com_O2                
cod         Lt     Mp      Td      Lt      Mp      Td
met                                                  
atp_c    1.010  1.010  1.0100  1.0100  1.0100  1.0100
ctp_c    0.000  0.000  0.0000  1.6833  1.6833  1.6833
dctp_c   0.000  0.000  0.0000  1.6833  1.6833  1.6833
dhorS_c  2.525  2.525  2.5250  2.5250  2.5250  2.5250
dhorS_m    NaN    NaN  1.2557     NaN     NaN  2.5250
dttp_c   0.000  0.000  0.0000  2.5250  2.5250  2.5250
gtp_c    1.010  1.010  1.0100  1.0100  1.0100  1.0100
orot_c   0.000  0.000  0.0000  2.5250  2.5250  2.5250
orot_m     NaN    NaN  1.1481     NaN     NaN  2.5250
ump_c    0.000  0.000  0.0000  2.5250  2.5250  2.5250
utp_c    0.000  0.000  0.0000  2.5250  2.5250  2.5250

quem produz o orotato:

Td:
   UF00665_CM   dhorS_c + q6_m --> orot_c + q6h2_m
                GPR=CCE94027
   UF00717_m    dhorS_m + fum_m --> orot_m + succ_m
                GPR=CCE94027
   UF00718_m    dhorS_m + o2_m --> h2o2_m + orot_m
   

## 10 · O que apareceu partido

A lista fechada dos defeitos medidos, por modelo. A coluna `nos_tres` é a que
distingue artefacto de template de propriedade do organismo.

In [25]:
df_ach = pd.DataFrame(ACHADOS)
if len(df_ach):
    print(df_ach.to_string(index=False))
    print("\nachados que aparecem nos tres modelos (artefacto do template):")
    for (teste, _), g in df_ach.groupby(["teste", df_ach.achado.str[:28]]):
        if len(set(g.codigo)) == 3:
            print(f"   [{teste}] {g.achado.iloc[0][:70]}")
    df_ach.to_csv(OUT / "achados.csv", index=False)
else:
    print("nenhum defeito detectado pelos nove testes.")

codigo teste                                                                                                                            achado   gravidade
    Td  T5.1                       formulas do par C18 iguais (ocda e 9Zocda ambos C18H34O2); 6 reaccoes com desvio de 1 protao dependem delas       grave
    Mp  T5.1                                                                                                                              idem       grave
    Lt  T5.1                                                                                                                              idem       grave
    Td    T8                                                                              indispensaveis: ash 1 g, iron(2+), dioxygen, sulfate informativo
    Mp    T8                                                                              indispensaveis: ash 1 g, iron(2+), dioxygen, sulfate informativo
    Lt    T8                                                          

## 11 · Curadoria

`scripts/curadoria_minima.py` é um catálogo novo, derivado apenas dos testes deste
notebook. Cada edição carrega três campos obrigatórios: `achado` (o teste que a
autoriza), `arbitro` (a referência que decide o sentido) e `variante` (onde se
aplica). Sem árbitro não há edição: uma alteração sem literatura é uma opinião.

Duas edições estão marcadas `aplicar_por_omissao = False` porque **criam
capacidade nova**, e isso exige uma decisão escrita e não um valor por omissão:

- **NS1** adiciona uma fumarato-redutase. Confirmar o homólogo de `OSM1`/`FRD1`
  nos três genomas antes de a aplicar. Sem ela nenhum dos três cresce em
  anaerobiose, mas isso é razão para verificar o genoma, não para a aplicar às
  cegas.
- **NS6** adiciona NGAM com um valor emprestado da *S. cerevisiae*. Fica de fora
  por omissão e declara-se nas limitações.

In [26]:
import curadoria_minima as CU

for cod in MODELOS:
    eds = CU.edicoes_de(cod, so_por_omissao=False)
    print(f"\n{'=' * 92}\n{cod} — {len(eds)} edicoes no catalogo\n{'=' * 92}")
    print(pd.DataFrame([{"id": e.id, "variante": e.variante,
                         "por_omissao": e.aplicar_por_omissao,
                         "titulo": e.titulo[:56]} for e in eds]).to_string(index=False))

print(f"\n{'=' * 92}\njustificacao de cada edicao\n{'=' * 92}")
for e in CU.edicoes_de("Td", so_por_omissao=False):
    print(f"\n[{e.id}] {e.titulo}\n   achado : {e.achado}\n   arbitro: {e.arbitro}")


Td — 8 edicoes no catalogo
  id  variante  por_omissao                                                   titulo
 NS0     ambas         True     Formulas do par C18 e os protoes que dependiam delas
 NS1 anaerobia        False       Reoxidacao anaerobia do quinol (fumarato-redutase)
NS2a     ambas         True                               Retinoato fora da biomassa
NS2b anaerobia         True      Cofactores respiratorios fora da biomassa anaerobia
 NS3 anaerobia         True Pool de esterol a 100 % ergosterol na variante anaerobia
 NS4     ambas         True           Lactato: sentido oxidativo e tecto de secrecao
 NS5     ambas         True                       Reducao de hexose a poliol fechada
 NS6     ambas        False           Manutencao nao associada ao crescimento (NGAM)

Mp — 8 edicoes no catalogo
  id  variante  por_omissao                                                   titulo
 NS0     ambas         True     Formulas do par C18 e os protoes que dependiam delas
 NS1 anae

In [27]:
# Chave = id da edicao, valor = o achado que a justifica. Uma edicao sem achado
# nao entra. Preenchido DEPOIS de ler a seccao 10.
EDICOES = {
    "NS0":  "T5.1 — ocda e 9Zocda ambos C18H34O2 nos tres; 6 reaccoes com desvio "
            "de 1 protao dependem das formulas",
    "NS2a": "T11 — retn_c so se fabrica com O2, e nao ha via de retinoides "
            "descrita em Saccharomycetaceae nem em Metschnikowia",
    "NS4":  "T13 — o carbono vai todo para lactato e o etanol fica a zero a "
            "todos os niveis de O2",
    "NS5":  "T13/T9 — sorbitol e manitol secretados como dreno redox no lugar "
            "do glicerol",
    # anaerobias
    "NS2b": "T11/T12 — ficytc_c, focytc_c e q6_m so se fabricam com O2",
    "NS3":  "T11 — sterol_1g_r bloqueado pelos 0.04 de zimosterol de UF02415_R",
    "NS1":  "T11 — dna_1g_c e rna_1g_c so se fabricam com O2: sem consumidor "
            "anaerobio de q6h2_m a DHOD UF00665_CM nao roda. DECISAO: aplicada "
            "a titulo provisorio, a aguardar confirmacao do homologo de "
            "OSM1/FRD1 nos tres genomas",
}

BASE = [e for e in EDICOES if CU.CATALOGO[e].variante in ("base", "ambas")]
ANA = [e for e in EDICOES if CU.CATALOGO[e].variante == "anaerobia"]
SEM_ANA = set(CU.SEM_ANAEROBIOSE)          # ver seccao 11.2
print("variante base     :", BASE)
print("variante anaerobia:", ANA, "| sem variante anaerobia:", sorted(SEM_ANA))

from cobra.flux_analysis import pfba

MU_ANTES, PERFIL_ANTES, CURADOS, ANAEROBIOS, LOGS = {}, {}, {}, {}, {}
for cod, m in MODELOS.items():
    MU_ANTES[cod] = {}
    for q in (0.0, 2.0):
        with m:
            meio_de_trabalho(m, cod, q_o2=q)
            v = m.slim_optimize()
            MU_ANTES[cod][q] = 0.0 if v != v else v
    with m:
        meio_de_trabalho(m, cod, q_o2=2.0)
        PERFIL_ANTES[cod] = pfba(m).fluxes.copy()

    mc = m.copy()
    mc.id = f"{m.id}_{cod}_curado"
    log = CU.aplicar(mc, cod, [e for e in BASE if cod in CU.CATALOGO[e].organismos],
                     verbose=False)
    CURADOS[cod] = mc

    if cod in SEM_ANA:
        LOGS[cod] = log
        pd.DataFrame(LOGS[cod]).to_csv(OUT / f"log_edicoes_{cod}.csv", index=False)
        print(f"{cod}: mu(O2=0) {MU_ANTES[cod][0.0]:.5f} | mu(O2=2) "
              f"{MU_ANTES[cod][2.0]:.5f} | {len(log)} operacoes base | "
              f"SEM variante anaerobia (seccao 11.2)")
        continue

    ma = mc.copy()
    ma.id = mc.id + "_anaerobio"
    log_a = CU.modo_anaerobio(ma, cod, o2_rid=MAPA[cod]["entradas"]["oxygen"],
                              ids=[e for e in ANA if cod in CU.CATALOGO[e].organismos])
    ANAEROBIOS[cod] = ma
    LOGS[cod] = log + log_a
    pd.DataFrame(LOGS[cod]).to_csv(OUT / f"log_edicoes_{cod}.csv", index=False)
    print(f"{cod}: mu(O2=0) {MU_ANTES[cod][0.0]:.5f} | mu(O2=2) "
          f"{MU_ANTES[cod][2.0]:.5f} | {len(log)} operacoes base + "
          f"{len(log_a)} anaerobias")

variante base     : ['NS0', 'NS2a', 'NS4', 'NS5']
variante anaerobia: ['NS2b', 'NS3', 'NS1'] | sem variante anaerobia: ['Mp']
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp0zii8rdp.lp
Reading time = 0.00 seconds
: 1510 rows, 3940 columns, 15096 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp_7xk6sek.lp
Reading time = 0.00 seconds
: 1510 rows, 3940 columns, 15088 nonzeros
Td: mu(O2=0) 0.00000 | mu(O2=2) 0.46837 | 24 operacoes base + 7 anaerobias
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp_iimwvk1.lp
Reading time = 0.00 seconds
: 1633 rows, 4262 columns, 16440 nonzeros
Mp: mu(O2=0) 0.00000 | mu(O2=2) 0.50679 | 20 operacoes base | SEM variante anaerobia (seccao 11.2)
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpl9es2go2.lp
Reading time = 0.01 seconds
: 1647 rows, 4276 columns, 16484 nonzeros
Read LP format model from file /var/fold

### 11.1 · O que cada edição mudou

Uma chamada de função e uma contagem não são uma curadoria legível. Para cada
modelo: as reações que a curadoria tocou, com a equação e os bounds antes e
depois, separadas por variante.

In [28]:
def diff(antes, depois, titulo):
    print("\n" + "=" * 92); print(titulo); print("=" * 92)
    for r in antes.reactions:
        if r.id not in depois.reactions:
            print(f"  {r.id} (REMOVIDA)"); continue
        novo = depois.reactions.get_by_id(r.id)
        if r.reaction != novo.reaction or r.bounds != novo.bounds:
            print(f"  {r.id}")
            print(f"     antes : {r.reaction[:110]}   {r.bounds}")
            print(f"     depois: {novo.reaction[:110]}   {novo.bounds}")
    for rid in sorted(set(depois.reactions.list_attr("id")) -
                      set(antes.reactions.list_attr("id"))):
        r = depois.reactions.get_by_id(rid)
        print(f"  {rid} (NOVA): {r.reaction}   {r.bounds}")

for cod in MODELOS:
    diff(MODELOS[cod], CURADOS[cod], f"{cod} — variante base (ficheiro exportado)")
    if cod in ANAEROBIOS:
        diff(CURADOS[cod], ANAEROBIOS[cod],
             f"{cod} — variante anaerobia (CU.modo_anaerobio)")
    else:
        print(f"\n{cod} — sem variante anaerobia, por decisao (seccao 11.2)")

print("\n\nlog completo:")
print(pd.concat({c: pd.DataFrame(l) for c, l in LOGS.items()}, names=["codigo"])
      .reset_index(level=0)[["codigo", "edicao", "op", "reaccao", "nota"]]
      .to_string(index=False))


Td — variante base (ficheiro exportado)
  UF00049_C
     antes : fru_c + h_c + nadph_c --> mnl_c + nadp_c   (0.0, 1000.0)
     depois: fru_c + h_c + nadph_c --> mnl_c + nadp_c   (0.0, 0.0)
  UF00103_C
     antes : lacL_c + nad_c <=> h_c + nadh_c + pyr_c   (-1000.0, 1000.0)
     depois: lacL_c + nad_c --> h_c + nadh_c + pyr_c   (0.0, 1000.0)
  UF01585_X
     antes : h2o_x + stcoa_x --> coa_x + 2.0 h_x + ocda_x   (0.0, 1000.0)
     depois: h2o_x + stcoa_x --> coa_x + h_x + ocda_x   (0.0, 1000.0)
  UF01591_C
     antes : 9Zocdcoa_c + h2o_c --> 9Zocda_c + coa_c   (0.0, 1000.0)
     depois: 9Zocdcoa_c + h2o_c --> 9Zocda_c + coa_c + h_c   (0.0, 1000.0)
  UF02328_C
     antes : 9Zocda_c + atp_c + coa_c --> 9Zocdcoa_c + amp_c + h_c + ppi_c   (0.0, 1000.0)
     depois: 9Zocda_c + atp_c + coa_c --> 9Zocdcoa_c + amp_c + ppi_c   (0.0, 1000.0)
  UF02328_X
     antes : 9Zocda_x + atp_x + coa_x --> 9Zocdcoa_x + amp_x + h_x + ppi_x   (0.0, 1000.0)
     depois: 9Zocda_x + atp_x + coa_x --> 9Zocdcoa_x 

### 11.2 · A *Metschnikowia pulcherrima* não tem variante anaeróbia

**Decisão de 2026-09-24. A Mp simula-se sempre com O₂ declarado.**

O raciocínio, por partes, porque é o tipo de decisão que parece arbitrária se só
se ler a conclusão.

**O que os testes mostraram.** No ficheiro por curar, nenhum dos três modelos
crescia a O₂ = 0. O T11 localizou sete componentes da biomassa que só se fabricam
com oxigénio, os mesmos sete nos três modelos, e o T12 confirmou que os quatro
cofactores envolvidos pesam menos de 0.07 mg/gDW cada. Sete bloqueios idênticos em
três espécies diferentes são assinatura do template do CarveFungi, não das
espécies. As edições NS1, NS2b e NS3 removem esses artefactos e os três passam a
crescer sem oxigénio.

**Onde é que isso corre mal.** Removidos os artefactos, a Mp fica com µ = 0.377 em
anaerobiose estrita, acima da Td (0.363) e da Lt (0.366), e com o rendimento em
etanol mais alto dos quatro (1.754 mol/mol). Ou seja: o modelo passa a dizer que a
*M. pulcherrima* é o melhor fermentador anaeróbio do consórcio. É o contrário do
que a espécie faz. A Mp é fracamente fermentativa, depende de oxigénio, morre cedo
na fermentação, e é usada em enologia precisamente para **reduzir** etanol, por
desvio respiratório de carbono sob arejamento controlado.

**A distinção que resolve isto.** Um bloqueio pode ser artefacto e o fenótipo que
ele produz estar, ainda assim, certo pelas razões erradas. É o caso. Os sete
bloqueios são artefactos do template, sim. Mas a ausência de crescimento anaeróbio
da Mp não é artefacto: é fenótipo documentado. Corrigir os artefactos e ficar com
crescimento anaeróbio é substituir um erro por outro, e o segundo é pior porque
parece um resultado.

Não se resolve pondo os bloqueios de volta, que seria curar para trás. Resolve-se
não simulando a Mp num regime em que a espécie não vive.

**O que fica decidido:**

1. A Mp exporta-se com as edições de variante base (`NS0`, `NS2a`, `NS4`, `NS5`),
   iguais às das outras duas. O ficheiro é o mesmo tipo de ficheiro.
2. A Mp **não** tem variante anaeróbia. `CU.modo_anaerobio(m, "Mp")` levanta
   `ValueError` com a justificação. A guarda está no código e não num comentário,
   para que nenhum script a jusante a possa pôr em anaerobiose por distração.
3. A Mp simula-se sempre com O₂ declarado, em `CU.Q_O2_DECLARADO["Mp"]`:
   `microaerobiose` = 0.5 e `arejamento` = 2.0 mmol/gDW/h. **Os dois valores são
   provisórios** e têm de ser amarrados ao protocolo de arejamento dos ensaios.
   Enquanto não houver protocolo, reporta-se o intervalo da curva abaixo e não um
   ponto.

**O que esta decisão não resolve.** A Mp continua a dar o rendimento em etanol mais
alto dos três a qualquer nível de O₂ baixo, e a razão disso não é a anaerobiose: é
não ter glicerol-3-fosfatase. Sem GPP não produz glicerol, e o glicerol é o dreno
redox que nas outras duas consome 3 a 4 % do carbono. Essa limitação está declarada
na secção 14 e não se cura aqui.

In [29]:
# Curva de O2 da Mp curada. E esta curva, e nao um valor, que se reporta enquanto
# nao houver protocolo de arejamento.
mp = CURADOS["Mp"]
et_mp = MAPA["Mp"]["saidas"]["ethanol"]
ac_mp = MAPA["Mp"]["saidas"].get("acetate")

linhas = []
for q in (0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 20.0):
    with mp:
        meio_de_trabalho(mp, "Mp", q_o2=q)
        v = mp.slim_optimize()
        if v != v or v < 1e-9:
            linhas.append({"O2": q, "cresce": False, "mu": 0.0}); continue
        s = pfba(mp); f = s.fluxes
        hexo = sum(-f[r.id] for r in mp.exchanges
                   if f.get(r.id, 0.0) < -1e-9
                   and (next(iter(r.metabolites)).elements or {}).get("C", 0) == 6)
        linhas.append({"O2": q, "cresce": True, "mu": round(f["BIOMASS"], 4),
                       "hexose": round(hexo, 2), "etanol": round(f.get(et_mp, 0.0), 2),
                       "etanol_por_hexose": round(f.get(et_mp, 0.0) / hexo, 3) if hexo > 1e-9 else None,
                       "acetato": round(f.get(ac_mp, 0.0), 2) if ac_mp else None})
d_mp = pd.DataFrame(linhas).fillna("-")
print("Mp curada — curva de O2 (mmol/gDW/h)")
print(d_mp.to_string(index=False))
d_mp.to_csv(OUT / "curva_O2_Mp.csv", index=False)

print(f"\nregimes declarados: {CU.Q_O2_DECLARADO['Mp']}")
print("referencia, anaerobiose estrita das outras duas:")
for cod in ("Td", "Lt"):
    if cod in ANAEROBIOS:
        with ANAEROBIOS[cod]:
            meio_de_trabalho(ANAEROBIOS[cod], cod, q_o2=0.0)
            print(f"   {cod}: mu = {ANAEROBIOS[cod].slim_optimize():.4f}")

achado("Mp", "11.2", "sem variante anaerobia por decisao: fracamente fermentativa "
                     "e dependente de O2. Simula-se em O2 declarado", "informativo")

Mp curada — curva de O2 (mmol/gDW/h)
   O2  cresce     mu hexose etanol etanol_por_hexose acetato
 0.00   False 0.0000      -      -                 -       -
 0.10    True 0.3838   20.0  35.14             1.757     0.0
 0.25    True 0.3966   20.0  35.01              1.75     0.0
 0.50    True 0.4169   20.0  34.81              1.74     0.0
 1.00    True 0.4495   20.0  34.17             1.708    0.21
 2.00    True 0.5068   20.0  32.54             1.626    1.11
 5.00    True 0.6757  20.01  27.66             1.383    3.83
20.00    True 0.9134  12.02    0.0               0.0     6.3

regimes declarados: {'microaerobiose': 0.5, 'arejamento': 2.0}
referencia, anaerobiose estrita das outras duas:
   Td: mu = 0.3626
   Lt: mu = 0.3656
  [Mp 11.2] sem variante anaerobia por decisao: fracamente fermentativa e dependente de O2. Simula-se em O2 declarado


## 12 · Os testes outra vez, sobre os modelos curados

Três perguntas, e as três têm de ter resposta antes de exportar:

1. O modelo **aeróbio** mudou? Se µ subir em aerobiose, foi criada capacidade nova
   e isso não estava em nenhuma edição.
2. O modelo **anaeróbio** cresce, e com que fisiologia? É o T13 e o T14 que
   respondem, não o µ sozinho.
3. A curadoria abriu algum ciclo de energia? T10 outra vez. A NS1 adiciona uma
   reação que consome quinol: é exatamente o tipo de edição que pode abrir um
   ciclo, e tem de ser verificada.

In [30]:
O2_MP = CU.Q_O2_DECLARADO["Mp"]["microaerobiose"]

res = []
for cod in MODELOS:
    for nome, m in (("antes", MODELOS[cod]), ("curado", CURADOS[cod])):
        for q in (0.0, 2.0, 20.0):
            with m:
                meio_de_trabalho(m, cod, q_o2=q)
                v = m.slim_optimize()
            res.append({"codigo": cod, "modelo": nome, "O2": q,
                        "mu": round(0.0 if v != v else v, 6)})
    if cod in ANAEROBIOS:
        with ANAEROBIOS[cod]:
            meio_de_trabalho(ANAEROBIOS[cod], cod, q_o2=0.0)
            v = ANAEROBIOS[cod].slim_optimize()
        res.append({"codigo": cod, "modelo": "anaerobio", "O2": 0.0,
                    "mu": round(0.0 if v != v else v, 6)})
    else:
        with CURADOS[cod]:
            meio_de_trabalho(CURADOS[cod], cod, q_o2=O2_MP)
            v = CURADOS[cod].slim_optimize()
        res.append({"codigo": cod, "modelo": f"microaerobio (O2={O2_MP})",
                    "O2": O2_MP, "mu": round(0.0 if v != v else v, 6)})
print(pd.DataFrame(res).pivot(index=["codigo", "modelo"], columns="O2",
                              values="mu").to_string())

print("\nT10 sobre os curados:")
print(pd.DataFrame({cod: T.t10_egc(m, T.PARES_NS)
                    for cod, m in CURADOS.items()}).T.to_string())
if ANAEROBIOS:
    print("\nT10 sobre os anaerobios (a NS1 adiciona um consumidor de quinol):")
    print(pd.DataFrame({cod: T.t10_egc(m, T.PARES_NS)
                        for cod, m in ANAEROBIOS.items()}).T.to_string())

print("\nT5 desequilibrio real:")
for cod in MODELOS:
    a = D5[cod]; a = a[~a.sem_formula] if len(a) else a
    b = T.t5_balancos(CURADOS[cod]); b = b[~b.sem_formula] if len(b) else b
    print(f"  {cod}: {len(a)} -> {len(b)}")

O2                                0.0       0.5       2.0       20.0
codigo modelo                                                       
Lt     anaerobio              0.365631       NaN       NaN       NaN
       antes                  0.000000       NaN  0.470041  0.913357
       curado                 0.000000       NaN  0.470065  0.913357
Mp     antes                  0.000000       NaN  0.506788  0.913357
       curado                 0.000000       NaN  0.506804  0.913357
       microaerobio (O2=0.5)       NaN  0.416943       NaN       NaN
Td     anaerobio              0.362565       NaN       NaN       NaN
       antes                  0.000000       NaN  0.468372  0.913357
       curado                 0.000000       NaN  0.468395  0.913357

T10 sobre os curados:
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp193jvoc7.lp
Reading time = 0.00 seconds
: 1510 rows, 3940 columns, 15088 nonzeros
Read LP format model from file /var/folders/pc/ykhcdw

### 12.1 · Fisiologia anaeróbia depois da curadoria

A pergunta que interessa ao projeto: com os modelos curados e em anaerobiose,
quanto etanol, quanto glicerol, quanto lactato, e o carbono fecha.

In [31]:
def fisiologia(m, cod, q_o2, rotulo):
    saidas = {k: v for k, v in MAPA[cod]["saidas"].items() if v in m.reactions}
    with m:
        meio_de_trabalho(m, cod, q_o2=q_o2)
        print(f"\n{'=' * 78}\n{cod} — {ESPECIE[cod]}, {rotulo}\n{'=' * 78}")
        d = T.t13_produtos(m, MAPA[cod]["entradas"]["oxygen"], saidas, niveis=(q_o2,))
        col = d.columns[0]
        print(d[d[col].astype(str) != "(0.0, 0.0)"].to_string())
        print(pd.Series(T.t14_carbono(m, BIOMASSA[cod],
                                      MAPA[cod]["saidas"]["ethanol"])).to_string())

for cod in MODELOS:
    if cod in ANAEROBIOS:
        fisiologia(ANAEROBIOS[cod], cod, 0.0, "anaerobiose estrita")
    else:
        for regime, q in CU.Q_O2_DECLARADO[cod].items():
            fisiologia(CURADOS[cod], cod, q, f"{regime}, O2 = {q} mmol/gDW/h")


Td — Torulaspora delbrueckii, anaerobiose estrita
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpg66jaaia.lp
Reading time = 0.00 seconds
: 1511 rows, 3943 columns, 15091 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp4vjukje6.lp
Reading time = 0.00 seconds
: 1511 rows, 3943 columns, 15091 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp_r_n9w86.lp
Reading time = 0.00 seconds
: 1511 rows, 3943 columns, 15091 nonzeros
Set parameter Username
Set parameter LicenseID to value 28

### 12.2 · Perfis de fluxo em aerobiose

Uma curadoria que só remove alternativas redundantes deve deixar µ igual e mudar
a distribuição de fluxos. A comparação usa pFBA dos dois lados: com FBA simples a
solução é um entre muitos ótimos, e a tabela acusaria centenas de reações
alteradas sem nada ter mudado no modelo.

In [32]:
for cod, mc in CURADOS.items():
    with mc:
        meio_de_trabalho(mc, cod, q_o2=2.0)
        sol = pfba(mc)
    comum = PERFIL_ANTES[cod].index.intersection(sol.fluxes.index)
    mudou = (sol.fluxes[comum] - PERFIL_ANTES[cod][comum]).abs()
    mudou = mudou[mudou > 1e-6].sort_values(ascending=False)
    print(f"\n{cod}: {len(mudou)} de {len(comum)} reaccoes com fluxo alterado")
    print(mudou.head(8).to_string())


Td: 283 de 1970 reaccoes com fluxo alterado
UF03474_E      33.761110
UF00022_M      33.748539
UF03263_E      33.748539
UF03155_TCM    33.748539
UF03263_TCM    33.748539
UF03263_TCE    33.748539
UF00103_C      33.748415
UF02967_E      33.748415

Mp: 256 de 2131 reaccoes com fluxo alterado
UF03474_E      32.549150
UF00242_C      32.536396
UF00309_C      32.536396
UF01871_C      32.536251
UF03245_TCE    32.535847
UF02971_E      32.535847
UF03263_TCE    32.535772
UF03263_E      32.535772

Lt: 276 de 2138 reaccoes com fluxo alterado
UF03474_E      31.940740
UF03263_TCE    31.928123
UF03263_E      31.928123
UF00022_C      31.928123
UF00103_C      31.928000
UF03352_TCE    31.928000
UF02967_E      31.928000
UF03227_TCE    31.927853


### 12.3 · Lactato contra etanol na Td: ótimos alternativos

A prova de que as colunas de secreção do T9 não eram fisiologia. No modelo por
curar, o FVA no ótimo estrito dá o mesmo intervalo ao etanol e ao lactato, e
fechar o lactato não altera µ. Foi esta célula que motivou a NS4.

In [33]:
for rid in ("UF03263_E", "UF02967_E", "UF02971_E", "UF03160_E"):
    met = next(iter(MODELOS["Td"].reactions.get_by_id(rid).metabolites))
    print(f"{rid}  {met.name}  {met.formula}")

UF03263_E  ethanol  C2H6O
UF02967_E  (S)-lactate  C3H5O3
UF02971_E  (R)-lactate  C3H5O3
UF03160_E  acetate  C2H3O2


In [34]:
from cobra.flux_analysis import flux_variability_analysis as fva

cod = "Td"
m = MODELOS[cod]
et  = MAPA[cod]["saidas"]["ethanol"]
lac = MAPA[cod]["saidas"]["L-lactate"]

with m:
    meio_de_trabalho(m, cod, q_o2=2.0)
    mu_livre = m.slim_optimize()
    # FVA ao otimo estrito: quanto pode variar cada produto SEM perder crescimento
    f = fva(m, reaction_list=[et, lac], fraction_of_optimum=1.0)
    print(f"mu = {mu_livre:.6f}")
    print(f.to_string())

    # e agora forcar a rota do etanol: fechar o lactato e ver se o mu se mantem
    m.reactions.get_by_id(lac).bounds = (0.0, 0.0)
    print(f"mu com lactato fechado = {m.slim_optimize():.6f}")

Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpwgnt955r.lp
Reading time = 0.00 seconds
: 1511 rows, 3941 columns, 15099 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp5peojlhk.lp
Reading time = 0.00 seconds
: 1511 rows, 3941 columns, 15099 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpmjkmoijr.lp
Reading time = 0.00 seconds
: 1511 rows, 3941 columns, 15099 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use onl

### 12.4 · T15 — rendimento em função do µ, e orçamento de carbono

A µ fixo os modelos ficam a poucos por cento uns dos outros, e a ordem só aparece
quando cada um corre ao seu próprio µmax. A biomassa do CarveFungi carrega
37.7 mmol C/gDW contra 41.1 do yeast-GEM, com GAM 60 contra 55.3 e sem NGAM: são
esses três parâmetros de template que fixam a ordem, não a espécie.

A Mp entra com o O₂ declarado da secção 11.2, não com zero. A comparação de redução
de etanol constrói-se no eixo do O₂ com captação de açúcar fixa, não no rendimento
anaeróbio.

In [35]:
ALVOS = {}
for cod in MODELOS:
    if cod in ANAEROBIOS:
        ALVOS[cod] = (ANAEROBIOS[cod], 0.0, "anaerobio")
    else:
        ALVOS[cod] = (CURADOS[cod], CU.Q_O2_DECLARADO[cod]["microaerobiose"],
                      f"microaerobio O2={CU.Q_O2_DECLARADO[cod]['microaerobiose']}")

D15 = {}
for cod, (m, q_o2, rotulo) in ALVOS.items():
    meio_cod = (lambda c: (lambda mm, q_o2=0.0: meio_de_trabalho(mm, c, q_o2=q_o2)))(cod)
    D15[cod] = T.t15_rendimento(m, BIOMASSA[cod], MAPA[cod]["saidas"]["ethanol"],
                                meio_cod, q_o2=q_o2)
    D15[cod].to_csv(OUT / f"T15_rendimento_{cod}.csv", index=False)
    print(f"{cod}: {rotulo}")

tab = pd.concat(D15, names=["codigo"]).reset_index(level=0)
print("\netanol por hexose (mol/mol), com o mu imposto:")
print(tab.pivot(index="mu_imposto", columns="codigo",
                values="etanol_por_hexose").to_string())
print("\nglicerol (mmol/gDW/h):")
print(tab.pivot(index="mu_imposto", columns="codigo", values="glicerol").to_string())

for cod, (m, q_o2, rotulo) in ALVOS.items():
    meio_cod = (lambda c: (lambda mm, q_o2=0.0: meio_de_trabalho(mm, c, q_o2=q_o2)))(cod)
    resumo, orc = T.t15_orcamento(m, BIOMASSA[cod], meio_cod, q_o2=q_o2)
    print(f"\n{cod} ({rotulo}): " + ", ".join(f"{k} = {v}" for k, v in resumo.items()))
    print(orc.to_string(index=False))
    orc.to_csv(OUT / f"T15_orcamento_{cod}.csv", index=False)

Td: anaerobio
Mp: microaerobio O2=0.5
Lt: anaerobio

etanol por hexose (mol/mol), com o mu imposto:
codigo         Lt     Mp     Td
mu_imposto                     
0.0         2.006  2.016  2.007
0.05        1.965  1.987  1.964
0.1         1.923  1.958  1.921
0.2         1.839  1.897  1.834
0.3         1.756  1.835  1.748
max         1.701  1.740  1.694

glicerol (mmol/gDW/h):
codigo        Lt   Mp    Td
mu_imposto                 
0.0         0.00  0.0  0.00
0.05        0.15  0.0  0.00
0.1         0.00  0.0  0.39
0.2         0.00  0.0  0.79
0.3         1.13  0.0  1.18
max         1.29  0.0  1.43

Td (anaerobio): mu = 0.3626, C_entra_mmol = 120.98, mmolC_por_gDW = 37.7
       destino  mmol_C  pct
       ethanol   67.78 56.0
carbon dioxide   34.70 28.7
      BIOMASSA   13.68 11.3
      glycerol    4.29  3.5
 succinate(2-)    0.52  0.4

Mp (microaerobio O2=0.5): mu = 0.4169, C_entra_mmol = 121.2, mmolC_por_gDW = 37.7
       destino  mmol_C  pct
       ethanol   69.62 57.4
carbon dioxide 

## 13 · Exportação

Um ficheiro por organismo, sem sufixo de variante. Dois ficheiros para o mesmo
organismo são duas fontes de verdade a atravessar todo o pipeline.

A variante anaeróbia **não** é um segundo ficheiro: quem precisar dela a jusante
chama `CU.modo_anaerobio(m, cod, o2_rid=..., ids=[...])` sobre o ficheiro curado,
e fica com o log das edições aplicadas nessa chamada. É o desenho do próprio
yeast-GEM, onde a anaerobiose é a função `anaerobicModel.m` e não um segundo XML.

In [36]:
NOME = {"Td": "model_delbrueckii_curado.xml",
        "Mp": "model_pulcherrima_curado.xml",
        "Lt": "model_thermotolerans_curado.xml"}

(ROOT / "models" / "curados").mkdir(parents=True, exist_ok=True)
for cod, mc in CURADOS.items():
    caminho = ROOT / "models" / "curados" / NOME[cod]
    cobra.io.write_sbml_model(mc, str(caminho))
    relido = cobra.io.read_sbml_model(str(caminho))
    with relido:
        meio_de_trabalho(relido, cod, q_o2=2.0)
        mu_aero = relido.slim_optimize()
    if cod in SEM_ANA:
        q = CU.Q_O2_DECLARADO[cod]["microaerobiose"]
        with relido:
            meio_de_trabalho(relido, cod, q_o2=q)
            mu_micro = relido.slim_optimize()
        print(f"{cod} -> {NOME[cod]}: {len(relido.reactions)} reaccoes | "
              f"aerobio mu = {mu_aero:.6f} | microaerobio (O2={q}) mu = {mu_micro:.6f}"
              f" | SEM variante anaerobia (seccao 11.2)")
        continue
    ana = relido.copy()
    CU.modo_anaerobio(ana, cod, o2_rid=MAPA[cod]["entradas"]["oxygen"],
                      ids=[e for e in ANA if cod in CU.CATALOGO[e].organismos])
    with ana:
        meio_de_trabalho(ana, cod, q_o2=0.0)
        mu_ana = ana.slim_optimize()
    print(f"{cod} -> {NOME[cod]}: {len(relido.reactions)} reaccoes | "
          f"aerobio mu = {mu_aero:.6f} | anaerobio mu = {mu_ana:.6f}")

print("\nA jusante: Td e Lt em anaerobiose por CU.modo_anaerobio; a Mp com O2 "
      "declarado em CU.Q_O2_DECLARADO['Mp']. CU.modo_anaerobio(m, 'Mp') levanta "
      "ValueError de proposito.")

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpxtnek05v.lp
Reading time = 0.00 seconds
: 1510 rows, 3940 columns, 15088 nonzeros
Td -> model_delbrueckii_curado.xml: 1970 reaccoes | aerobio mu = 0.468395 | anaerobio mu = 0.362565
Mp -> model_pulcherrima_curado.xml: 2131 reaccoes | aerobio mu = 0.506804 | microaerobio (O2=0.5) mu = 0.416943 | SEM variante anaerobia (seccao 11.2)
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpqlc8065v.lp
Reading time = 0.00 seconds
: 1647 rows, 4276 columns, 16476 nonzeros
Lt -> model_thermotolerans_curado.xml: 2138 reaccoes | aerobio mu = 0.470065 | anaerobio mu = 0.365631

A jusante: Td e Lt em anaerobiose por CU.modo_anaerobio; a Mp com O2 declarado em CU.Q_O2_DECLARADO['Mp']. CU.modo_anaerobio(m, 'Mp') levanta ValueError de proposito.


## 14 · Interpretação e limitações

*Rascunho preenchido com os números da execução de 2026-09-24. Reconfirmar depois
de correr o notebook de ponta a ponta.*

1. **As contagens do T1 são quase iguais nos três** e a equação de biomassa é
   idêntica termo a termo: mesmo template do CarveFungi. Três espécies com a mesma
   composição celular é limitação a declarar, não resultado.
2. **Não há fuga de massa nem ciclo de energia** (T4 e T10), antes nem depois da
   curadoria, incluindo a variante anaeróbia com a fumarato-redutase da NS1.
3. **As fórmulas do par C18 estavam erradas nos três** (estearato e oleato ambos
   C18H34O2) e seis reações tinham a estequiometria de protões escrita contra
   elas. Corrigido pela NS0: o desequilíbrio real desce 50→48 na Td, 65→64 na Mp e
   59→58 na Lt. Nenhum dos restantes atravessa a membrana plasmática.
4. **A anaerobiose estava bloqueada por quatro coisas ao mesmo tempo**, e nenhuma
   delas sozinha explica o µ = 0:
   - `retn_c`, `ficytc_c`, `focytc_c` e `q6_m` na biomassa, com 10⁻⁴ de
     coeficiente cada, todos dependentes de O₂ (NS2a e NS2b);
   - os 0.04 de zimosterol em `UF02415_R`: 4 % do pool de esterol a bloquear os
     100 % (NS3);
   - a ausência de consumidor anaeróbio de `q6h2_m`, que trava a DHOD
     `UF00665_CM` e com ela as pirimidinas, o RNA e o DNA (NS1).
5. **Nenhum dos três produzia etanol a nenhum nível de O₂.** A capacidade existia
   (FVA até 33.7 na Td), mas todo o carbono ia para lactato, e a célula 12.3
   mostra que fechar o lactato não muda µ: eram ótimos alternativos e a pFBA
   escolhia por número de passos. Com a NS4 e a NS5 o etanol passa a obrigatório
   (mínimo do FVA acima de zero) nos três, a 80–88 % do máximo teórico.
6. **Decisões espécie a espécie, e porquê:**
   - **Td**: rota redutora `pyr → D-lactato` fechada e L-lactato sem secreção.
     A *T. delbrueckii* distingue-se por baixa acidez volátil, não por lactato.
   - **Lt**: a LDH `UF00103_C` **fica**, com teto de secreção de 1.8 mmol/gDW/h
     (0.09 mol lactato/mol hexose, dos ~9 g/L descritos em vinificação). É a única
     das três com produção de ácido L-láctico documentada.
   - **Mp**: o D-lactato vinha da glioxalase `UF01603_C`, via de destoxificação do
     metilglioxal, não de uma LDH. Fechou-se a secreção.

7. **A Mp não tem variante anaeróbia**, por decisão registada na secção 11.2. Os
   sete bloqueios são artefactos do template, mas a ausência de crescimento
   anaeróbio da *M. pulcherrima* é fenótipo documentado: é fracamente
   fermentativa, depende de O₂ e é usada em enologia para reduzir etanol por
   desvio respiratório sob arejamento. Removidos os artefactos, o modelo passava a
   dizer que é o melhor fermentador anaeróbio do consórcio. Simula-se sempre com
   O₂ declarado, em `CU.Q_O2_DECLARADO["Mp"]`, e `CU.modo_anaerobio(m, "Mp")`
   levanta `ValueError` de propósito.

**Limitações conhecidas destes ficheiros, a declarar:**

- O pseudo-metabolito `ash` não tem fórmula nem teto derivável de uma receita.
  Fica aberto, conta zero carbono no T14, e no eixo mineral estes três modelos
  não são comparáveis com a *S. cerevisiae*, que troca K, Mg, Ca, Mn e Zn
  separadamente.
- **A Mp não tem glicerol-3-fosfatase.** `UF01626_C` existe na Td e na Lt e falta
  na Mp; a GPD está lá, a GPP não. O modelo não produz glicerol em cenário nenhum,
  e o glicerol é o dreno redox anaeróbio da levedura. Não se adicionou a reação
  porque isso é criar capacidade: procurar o homólogo de `GPP1`/`GPP2` no genoma
  da Mp antes de decidir.
- **Falta NGAM nos três.** Não há reação com `lower_bound > 0` envolvendo ATP, e
  não existe `ATPM`; só há GAM (60 no `BIOMASS`). Sem manutenção, a dFBA a jusante
  não tem fase estacionária, que é onde o vinho se faz. A edição NS6 existe no
  catálogo com o valor do yeast-GEM (0.7 mmol ATP/gDW/h) mas fica de fora por
  omissão: é um valor emprestado de outra espécie.
- **A Td não tem álcool-desidrogenase citosólica.** `UF00022_C` falta na Td e
  existe na Mp e na Lt. O etanol continua a sair pela via mitocondrial, com
  transporte presente, e µ não muda: é implausibilidade de compartimentação sem
  consequência numérica. Não se curou.
- **A via de Ehrlich está incompleta nos três**: sem trocas para 2-feniletanol,
  triptofol, tirosol nem propanol. Bloqueia trabalho sobre aroma e torna o SMETANA
  cego a essa classe de interações.
- A *M. pulcherrima* é a que dá µ mais alto dos três com tudo aberto, e é das mais
  lentas em cultura. O µ com tudo aberto não é uma taxa de crescimento: sem tetos
  de captação o modelo consome o que quiser.

**Ainda por fazer neste objetivo, antes do SMETANA:**

- Harmonizar os identificadores do compartimento extracelular entre estes três
  modelos e o yeast-GEM do `obj1a`. O SMETANA agrupa o meio partilhado por string
  do identificador da troca; com `UF*_E` aqui e `r_*` lá, os pools ficam separados
  e nenhuma interação entre famílias de modelos é detetada.
- Resolver o eixo mineral (`ash` contra as trocas separadas do yeast-GEM) antes de
  comparar µ ou rendimentos entre as duas famílias.
- Confirmar no genoma o homólogo de `OSM1`/`FRD1` (NS1) e de `GPP1`/`GPP2` (Mp).

## 15 · Bibliografia

1. Zorrilla F, Buric F, Patil KR, Zelezniak A (2021). metaGEM. *Nucleic Acids Research* 49(21):e126.
2. Lu H, Li F, Sánchez BJ, et al. (2019). A consensus *S. cerevisiae* metabolic model Yeast8. *Nature Communications* 10:3586.
3. SysBioChalmers/yeast-GEM, `code/otherChanges/anaerobicModel.m` — protocolo anaeróbio de referência: zera o heme a no pseudo-componente de cofactores, abre esteróis e ácidos gordos insaturados, fecha o O₂.
4. Fritzemeier CJ, Hartleb D, Szappanos B, Papp B, Lercher MJ (2017). Erroneous energy-generating cycles in published genome scale metabolic networks. *PLoS Computational Biology* 13(4):e1005494.
5. Chan SHJ, Cai J, Wang L, Simons MN, Maranas CD (2017). Standardizing biomass reactions and ensuring complete mass balance in genome-scale metabolic models. *Bioinformatics* 33(22):3603–3609.
6. Lieven C, Beber ME, Olivier BG, et al. (2020). MEMOTE for standardized genome-scale metabolic model testing. *Nature Biotechnology* 38:272–276.
7. Lewis NE, Hixson KK, Conrad TM, et al. (2010). Omic data from evolved *E. coli* are consistent with computed optimal growth from genome-scale models (pFBA). *Molecular Systems Biology* 6:390.
8. Nagy M, Lacroute F, Thomas D (1992). Divergent evolution of pyrimidine biosynthesis between anaerobic and aerobic yeasts. *PNAS* 89:8966–8970. — Ura1 é uma DHOD citosólica de classe 1A dependente de fumarato, e é por isso que a *S. cerevisiae* sintetiza pirimidinas sem cadeia respiratória.
9. Bouwknegt J, Wiersma SJ, Ortiz-Merino RA, et al. (2021). Class-II dihydroorotate dehydrogenases from three phylogenetically distant fungi support anaerobic pyrimidine biosynthesis. *Fungal Biology and Biotechnology* 8:10. — o artefacto não é a DHOD ser de classe II, é estar acoplada a um quinol que só se reoxida pela cadeia respiratória.
10. Arikawa Y, Enomoto K, Muratsubaki H, Okazaki M (1998). Soluble fumarate reductase isoenzymes from *Saccharomyces cerevisiae* are required for anaerobic growth. *FEMS Microbiology Letters* 165:111–116.
11. Camarasa C, Faucet V, Dequin S (2007). Role in anaerobiosis of the isoenzymes for *Saccharomyces cerevisiae* fumarate reductase encoded by *OSM1* and *FRDS1*. *Yeast* 24:391–401.
12. Andreasen AA, Stier TJB (1953, 1954). Anaerobic nutrition of *Saccharomyces cerevisiae*. *Journal of Cellular and Comparative Physiology* 41:23–36; 43:271–281.
13. Ansell R, Granath K, Hohmann S, Thevelein JM, Adler L (1997). The two isoenzymes for yeast NAD⁺-dependent glycerol 3-phosphate dehydrogenase, Gpd1p and Gpd2p. *EMBO Journal* 16:2179–2187.
14. Nissen TL, Schulze U, Nielsen J, Villadsen J (1997). Flux distributions in anaerobic, glucose-limited continuous cultures of *Saccharomyces cerevisiae*. *Microbiology* 143:203–218.
15. Benito S (2018). The impact of *Torulaspora delbrueckii* yeast in winemaking. *Applied Microbiology and Biotechnology* 102:3081–3094.
16. Morata A, Loira I, Escott C, et al. (2019). Applications of *Metschnikowia pulcherrima* in wine biotechnology. *Fermentation* 5(3):63.
17. Vilela A (2018). *Lachancea thermotolerans*, the non-*Saccharomyces* yeast that reduces the volatile acidity of wines. *Fermentation* 4(3):56.
18. Hranilovic A, Albertin W, Capone DL, et al. (2021). An integrative view of the role of *Lachancea thermotolerans* in wine technology. *Foods* 10(11):2578. — variabilidade da produção de ácido L-láctico entre estirpes.
19. Vaquero C, Loira I, Raso J, et al. (2022). Ethanol-lactate transition of *Lachancea thermotolerans* is linked to nitrogen metabolism. *Food Microbiology* 106:104049.